In [27]:
import os
import pandas as pd
from docx import Document
from collections import defaultdict

## Importing the Data

In [2]:
DATASET_PATH = "dataset"

In [3]:
def extract_text_from_docx(file_path):
    """
    Extracts text from a .docx file by reading all paragraphs.
    Returns the full text as a single string.
    """
    try:
        doc = Document(file_path)
        paragraphs = [para.text.strip() for para in doc.paragraphs if para.text.strip()]
        return "\n".join(paragraphs)
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return ""

In [4]:
resume_data = []

for filename in os.listdir(DATASET_PATH):
    if filename.lower().endswith(".docx"):
        file_path = os.path.join(DATASET_PATH, filename)
        text = extract_text_from_docx(file_path)

        resume_data.append({
            "filename": filename,
            "raw_text": text
        })

df_resumes = pd.DataFrame(resume_data)

In [5]:
df_resumes.head()

,filename,raw_text
0,Venkat_BA.docx,Venkat N\nPhone: 314-662-2902\nEmail: vhealthb...
1,Shashank.docx,SHASHANK TIWARI\nShashank.tiwari44@gmail.com ...
2,Anudeep N_Sr Java Developer.docx,Anudeep\nSr Java Programmer\nanudeepreddynalla...
3,ram nandyala.docx,RESUME\nRAMA KARTHIK NANDYALA\nMOBILE NO: (414...
4,Neha Mugghala.docx,Sourya\nSenior Java/J2EE Developer\nsny.java@g...


### Investigating the Data

In [6]:
print("Number of resumes:", len(df_resumes))
print("\nColumns:", df_resumes.columns.tolist())
print("\nMissing values:\n", df_resumes.isnull().sum())

Number of resumes: 228

Columns: ['filename', 'raw_text']

Missing values:
 filename    0
raw_text    0
dtype: int64


In [7]:
# Preview 1 resume
print(df_resumes.loc[0, "raw_text"][:3000])  # preview first 3000 characters

Venkat N
Phone: 314-662-2902
Email: vhealthba@gmail.com
Business Analyst
Summary:
Over 9+ Years of experience in Business Analysis, Design, Software Testing, Product Configuration, Project Co-ordination with Ender user training.
Have experience in Post production and Support which includes Training to the end users and preparation of End user manual and Business Process Procedures (BPP)
Adept in analyzing information system needs, evaluating end-user requirements, custom designing solutions for complex information systems management.
Extensive experience in Lead position with expertise in Requirements Gathering, Analysis, Testing coordination etc. Interface with business users to assess business needs and create business requirements well within timeframe exceeding the expectations.
Practical knowledge of SDLC methodologies like Waterfall, Spiral, ASAP and Agile.
Worked extensively on Oracle Cloud HCM(Core HR, Benefits, Payroll, compensation, talent management)
Experienced in the docum

In [8]:
df_resumes

,filename,raw_text
0,Venkat_BA.docx,Venkat N\nPhone: 314-662-2902\nEmail: vhealthb...
1,Shashank.docx,SHASHANK TIWARI\nShashank.tiwari44@gmail.com ...
2,Anudeep N_Sr Java Developer.docx,Anudeep\nSr Java Programmer\nanudeepreddynalla...
3,ram nandyala.docx,RESUME\nRAMA KARTHIK NANDYALA\nMOBILE NO: (414...
4,Neha Mugghala.docx,Sourya\nSenior Java/J2EE Developer\nsny.java@g...
...,...,...
223,Harshitha Challa.docx,Name: Harshitha ...
224,Venkata_Sr.PHP_Developer.docx,Venkata\nSr. PHP / Drupal Developer\nEmail: ve...
225,V Ashok A.docx,Ashok A\nashtechjava12@gmail.com\n(617) 681-42...
226,Navneeth Resume.docx,Navneet Gupta\nproject/ program manager (PMP® ...


### Cleaning the Data

In [9]:
# Check for empty resume
df_resumes["raw_text"] = df_resumes["raw_text"].fillna("").astype(str)

empty_resumes = df_resumes[df_resumes["raw_text"].str.strip() == ""]
print("Number of empty resumes:", len(empty_resumes))
empty_resumes.head()

Number of empty resumes: 4


,filename,raw_text
64,"Resume Vishal PM - MSIS, PMP-PMI.docx",
88,employer_mounika details.docx,
128,SAURABH_PM.docx,
189,Raju Goduguchinta_Technical Program Manager.docx,


In [10]:
# Delete the empty resume
df_resumes = df_resumes[df_resumes["raw_text"].str.strip() != ""].reset_index(drop=True)

print("Shape after removing empty resumes:", df_resumes.shape)

Shape after removing empty resumes: (224, 2)


In [13]:
#
df_resumes["char_length"] = df_resumes["raw_text"].apply(len)
df_resumes["word_count"] = df_resumes["raw_text"].apply(lambda x: len(x.split()))

df_resumes[["char_length", "word_count"]].describe()

,char_length,word_count
count,224.000000,224.000000
mean,18438.151786,2559.531250
std,7118.218578,1004.490732
min,2787.000000,368.000000
25%,13634.750000,1863.000000
50%,17830.500000,2462.500000
75%,22820.500000,3177.750000
max,51470.000000,6589.000000


In [14]:
import re

def normalize_for_duplicate_check(text):
    text = str(text).lower()
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    return text

# Create normalized version for duplicate detection
df_resumes["normalized_text"] = df_resumes["raw_text"].apply(normalize_for_duplicate_check)

# Count duplicates
duplicate_texts = df_resumes["normalized_text"].duplicated().sum()
print("Duplicate resume texts (after normalization):", duplicate_texts)

# Preview duplicates
duplicate_rows = df_resumes[df_resumes["normalized_text"].duplicated(keep=False)]
print("Number of duplicated resume entries:", len(duplicate_rows))

duplicate_rows[["filename", "word_count"]].sort_values("filename").head(20)

Duplicate resume texts (after normalization): 2
Number of duplicated resume entries: 4


,filename,word_count
115,Dave.docx,3903
61,Derik.docx,3903
88,Vishnu J.docx,2396
223,Vishnu Java dev.docx,2396


In [15]:
# Delete duplicated resume
df_resumes = df_resumes.drop_duplicates(subset="normalized_text").reset_index(drop=True)

print("Shape after removing duplicate resumes:", df_resumes.shape)

Shape after removing duplicate resumes: (222, 5)


In [16]:
# Check for very short resume
short_resumes = df_resumes[df_resumes["word_count"] < 50]
print("Number of very short resumes (< 50 words):", len(short_resumes))

short_resumes[["filename", "word_count"]].head(10)

Number of very short resumes (< 50 words): 0


,filename,word_count


## Text Processing

In [42]:
def clean_resume_text(text):
    """
    Safely clean resume text for NLP while preserving technical/job-related information.
    """
    text = str(text)

    # Remove HTML non-breaking space artifacts
    text = text.replace("&nbsp;", " ")
    text = text.replace("nbsp", " ")

    # Lowercase
    text = text.lower()

    # Replace common bullet characters with space
    text = re.sub(r'[•●▪◦■□◆▶►]', ' ', text)

    # Normalize line breaks and tabs into spaces
    text = re.sub(r'[\n\r\t]+', ' ', text)

    # Remove weird repeated spaces
    text = re.sub(r'\s+', ' ', text)

    # Keep useful technical punctuation only
    text = re.sub(r'[^a-z0-9\s\+\#@\./\-]', ' ', text)

    # Normalize repeated spaces again
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [43]:
# Cleaning the text
df_resumes["clean_text"] = df_resumes["raw_text"].apply(clean_resume_text)

In [44]:
# Preview sample cleaned text
sample_idx = 0

print("="*80)
print("RAW TEXT")
print("="*80)
print(df_resumes.loc[sample_idx, "raw_text"][:2000])

print("\n" + "="*80)
print("CLEANED TEXT")
print("="*80)
print(df_resumes.loc[sample_idx, "clean_text"][:2000])

RAW TEXT
Venkat N
Phone: 314-662-2902
Email: vhealthba@gmail.com
Business Analyst
Summary:
Over 9+ Years of experience in Business Analysis, Design, Software Testing, Product Configuration, Project Co-ordination with Ender user training.
Have experience in Post production and Support which includes Training to the end users and preparation of End user manual and Business Process Procedures (BPP)
Adept in analyzing information system needs, evaluating end-user requirements, custom designing solutions for complex information systems management.
Extensive experience in Lead position with expertise in Requirements Gathering, Analysis, Testing coordination etc. Interface with business users to assess business needs and create business requirements well within timeframe exceeding the expectations.
Practical knowledge of SDLC methodologies like Waterfall, Spiral, ASAP and Agile.
Worked extensively on Oracle Cloud HCM(Core HR, Benefits, Payroll, compensation, talent management)
Experienced in 

In [45]:
df_resumes["clean_word_count"] = df_resumes["clean_text"].apply(lambda x: len(x.split()))
df_resumes[["word_count", "clean_word_count"]].head()

,word_count,clean_word_count
0,2478,2479
1,1488,1491
2,1726,1707
3,2895,2903
4,2791,2785


## Resume Segmentation

In [46]:
SECTION_PATTERNS = {
    "summary": [
        "summary", "professional summary", "profile", "objective", "about me"
    ],
    "skills": [
        "skills", "technical skills", "core competencies", "competencies", "expertise"
    ],
    "experience": [
        "experience", "work experience", "professional experience", "employment history",
        "internship", "internships"
    ],
    "education": [
        "education", "academic background", "academic qualification", "qualifications"
    ],
    "projects": [
        "projects", "academic projects", "personal projects", "project experience"
    ],
    "certifications": [
        "certifications", "certificates", "licenses", "courses", "training"
    ],
    "achievements": [
        "achievements", "awards", "honors", "accomplishments"
    ]
}

In [47]:
# Heading normalization function
def normalize_heading(text):
    text = text.lower().strip()
    text = re.sub(r'[^a-z\s]', '', text)  # keep letters and spaces only
    text = re.sub(r'\s+', ' ', text)
    return text

In [48]:
# Detect whether a line is a section heading
def detect_section_heading(line, section_patterns):
    """
    Returns the section name if the line matches a known heading.
    Otherwise returns None.
    """
    normalized_line = normalize_heading(line)

    for section, headings in section_patterns.items():
        if normalized_line in headings:
            return section

    return None

In [49]:
# Segmentation Function
def segment_resume_sections(raw_text, section_patterns):
    """
    Segment resume text into sections based on detected headings.
    Returns a dictionary of sections.
    """
    sections = defaultdict(list)
    current_section = "other"  # default bucket

    lines = raw_text.split("\n")

    for line in lines:
        stripped_line = line.strip()

        if not stripped_line:
            continue

        detected_section = detect_section_heading(stripped_line, section_patterns)

        if detected_section:
            current_section = detected_section
        else:
            sections[current_section].append(stripped_line)

    # Join lists into text blocks
    segmented_output = {section: "\n".join(content).strip() for section, content in sections.items()}

    return segmented_output

In [50]:
sample_idx = 31

sample_sections = segment_resume_sections(df_resumes.loc[sample_idx, "raw_text"], SECTION_PATTERNS)

sample_sections.keys()

dict_keys(['other', 'experience', 'skills'])

In [51]:
# Preview segmented sections
for section, content in sample_sections.items():
    print("="*100)
    print(f"SECTION: {section.upper()}")
    print("="*100)
    print(content[:1500])  # preview first 1500 chars
    print("\n")

SECTION: OTHER
Sivanaga G
571-346-2112
sivagatiganti@gmail.com


SECTION: EXPERIENCE
8+ years of experience in development and implementation of Web-based Client-Server applications using Java and J2EE technologies.
Working knowledge in multi-tiered distributed environment, OOAD concepts, good understanding of Software Development Lifecycle (SDLC) and familiarity of Service Oriented Architecture (SOA).
Proficient in OOAD Technologies developing Use Cases, Activity diagrams, Sequence Diagrams and Class diagrams using case tools like Microsoft Visio and Rational Rose.
Experience in developing Web based GUI’s using JSP, HTML, DHTML, CSS, JavaScript (and its frameworks like JSON), Action Script, DOJO,Node JS, Angular JS, JQuery, EXT JS and Ajax.
Used JavaScript for client side validations and implemented AJAX with JavaScript for reducing data transfer overhead between user and server.
Expertise in application development using JSP, Servlets, JDBC, JNDI, Spring, Hibernate, JSF, EJB2.0/3.0, 

In [52]:
# Apply segmentation to all resumes
df_resumes["segmented_sections"] = df_resumes["raw_text"].apply(
    lambda x: segment_resume_sections(x, SECTION_PATTERNS)
)

In [53]:
# Extract major section columns
target_sections = ["summary", "skills", "experience", "education", "projects", "certifications", "achievements", "other"]

for section in target_sections:
    df_resumes[section] = df_resumes["segmented_sections"].apply(lambda x: x.get(section, ""))

In [54]:
# Preview section columns
df_resumes[[
    "filename",
    "skills",
    "education",
    "experience",
    "projects"
]].head(10)

,filename,skills,education,experience,projects
0,Venkat_BA.docx,"Databases: MS SQL Server, M...",Bachelor of Engineering\nWork Experience :\nAs...,,
1,Shashank.docx,,Masters -Business Administration (MBA) ...,"Goldman Sachs, New York (Client)\nBusiness Ana...",
2,Anudeep N_Sr Java Developer.docx,,Bachelor of Engineering in Computer Science an...,"Wind Stream Communication, Dallas, TX ...",
3,ram nandyala.docx,,Bachelor of Technology in Computer Science fro...,Client6 ...,
4,Neha Mugghala.docx,"Programming Languages: Java J2SE 1.6/1.7, J2EE...",Bachelors in Computer Science and Information ...,"AT&T, Alpharetta, GA \t\t\t\t\t\t ...",
5,chenna kesava.docx,,,"Client: Staples, Framingham, MA \t\t\t\t\t\tDu...",
6,Utthan Silawal12.docx,,,"Barneys - New York, NY\t\t\t\t\t\t ...",
7,Yohan BSA.docx,,,Sr. Business Systems Analyst\nNXP Semiconducto...,
8,Madhu_BA_AW.DOCX,,,"Chicago Board of Options Exchange(CBOE), Chica...",
9,KIRAN KUMAR.docx,,Jawaharlal Nehru Technological University\n2008,"Client: Genentech, CA\nRole: Java Full Stack D...",


## Feature Extraction

### Contact extraction

In [56]:
def extract_email(text):
    """
    Extract first email address found in the text.
    """
    text = str(text)
    match = re.search(r'[\w\.-]+@[\w\.-]+\.\w+', text)
    return match.group(0) if match else ""

In [57]:
def extract_phone(text):
    """
    Extract first phone number found in the text.
    Handles common international and local formats.
    """
    text = str(text)

    phone_pattern = r'(\+?\d[\d\s\-\(\)]{8,}\d)'
    match = re.search(phone_pattern, text)

    if match:
        return match.group(0).strip()
    return ""

In [58]:
df_resumes["email"] = df_resumes["raw_text"].apply(extract_email)
df_resumes["phone"] = df_resumes["raw_text"].apply(extract_phone)

In [59]:
df_resumes[["filename", "email", "phone"]].head(10)

,filename,email,phone
0,Venkat_BA.docx,vhealthba@gmail.com,314-662-2902
1,Shashank.docx,Shashank.tiwari44@gmail.com,650) 600-1785
2,Anudeep N_Sr Java Developer.docx,anudeepreddynallamada@gmail.com,
3,ram nandyala.docx,NANDYALARAMAKARTHIK1@GMAIL.COM,414)-858-6880
4,Neha Mugghala.docx,sny.java@gmail.com,
5,chenna kesava.docx,chennakesava231@gmail.com,469-677-7837
6,Utthan Silawal12.docx,usilawal123@gmail.com,707) 653-6269
7,Yohan BSA.docx,,
8,Madhu_BA_AW.DOCX,bnsmadhavi@gmail.com,908-313-0336
9,KIRAN KUMAR.docx,kumarjava174@gmail.com,510-770-6277


### Skills extraction

In [60]:
SKILL_KEYWORDS = [
    # Programming Languages
    "python", "java", "c", "c++", "c#", "javascript", "typescript", "php", "ruby", "go", "kotlin", "swift", "r", "matlab",

    # Web / Backend / Frontend
    "html", "css", "react", "react.js", "node.js", "nodejs", "express", "django", "flask", "laravel", "spring", "fastapi",

    # Data / ML / AI
    "machine learning", "deep learning", "data science", "tensorflow", "pytorch", "scikit-learn", "opencv", "nlp", "computer vision", "pandas", "numpy",

    # Databases
    "sql", "mysql", "postgresql", "postgres", "mongodb", "sqlite", "firebase",

    # Tools / Platforms
    "git", "github", "docker", "linux", "aws", "azure", "gcp", "figma", "tableau", "power bi", "excel", "jira",

    # Mobile / Others
    "android", "flutter", "dart", "unity", "rest api", "graphql"
]

In [61]:
def extract_skills(text, skill_keywords):
    """
    Extract matching skills from text using keyword lookup.
    Returns a sorted unique list.
    """
    text = str(text).lower()
    found_skills = set()

    for skill in skill_keywords:
        # Escape special characters like +, #
        pattern = r'\b' + re.escape(skill.lower()) + r'\b'
        if re.search(pattern, text):
            found_skills.add(skill.lower())

    return sorted(found_skills)

In [62]:
df_resumes["skill_source_text"] = (
    df_resumes["skills"].fillna("") + " " +
    df_resumes["projects"].fillna("") + " " +
    df_resumes["experience"].fillna("")
)

df_resumes["extracted_skills"] = df_resumes["skill_source_text"].apply(
    lambda x: extract_skills(x, SKILL_KEYWORDS)
)

In [63]:
df_resumes[["filename", "extracted_skills"]].head(10)

,filename,extracted_skills
0,Venkat_BA.docx,"[excel, sql]"
1,Shashank.docx,"[excel, sql]"
2,Anudeep N_Sr Java Developer.docx,"[c, css, github, html, java, javascript, linux..."
3,ram nandyala.docx,"[aws, css, docker, express, html, java, javasc..."
4,Neha Mugghala.docx,"[aws, css, git, html, java, javascript, jira, ..."
5,chenna kesava.docx,"[aws, css, docker, git, github, html, java, ja..."
6,Utthan Silawal12.docx,"[android, c, go, jira, sql]"
7,Yohan BSA.docx,"[excel, html, jira, mysql, power bi, rest api,..."
8,Madhu_BA_AW.DOCX,"[c, docker, excel, git, go, html, java, jira, ..."
9,KIRAN KUMAR.docx,"[aws, css, git, html, java, javascript, jira, ..."


### EDUCATION EXTRACTION

In [65]:
EDUCATION_KEYWORDS = [
    "bachelor", "master", "phd", "diploma", "associate",
    "computer science", "informatics", "information systems",
    "software engineering", "data science", "artificial intelligence",
    "gpa", "university", "college", "institute"
]

In [66]:
def extract_keywords(text, keyword_list):
    """
    Generic keyword extractor.
    Returns matched keywords from text.
    """
    text = str(text).lower()
    found = set()

    for keyword in keyword_list:
        pattern = r'\b' + re.escape(keyword.lower()) + r'\b'
        if re.search(pattern, text):
            found.add(keyword.lower())

    return sorted(found)

In [67]:
df_resumes["education_keywords"] = df_resumes["education"].apply(
    lambda x: extract_keywords(x, EDUCATION_KEYWORDS)
)

In [68]:
df_resumes[["filename", "education", "education_keywords"]].head(5)

,filename,education,education_keywords
0,Venkat_BA.docx,Bachelor of Engineering\nWork Experience :\nAs...,"[bachelor, master]"
1,Shashank.docx,Masters -Business Administration (MBA) ...,"[bachelor, college, diploma, institute, master..."
2,Anudeep N_Sr Java Developer.docx,Bachelor of Engineering in Computer Science an...,"[bachelor, computer science]"
3,ram nandyala.docx,Bachelor of Technology in Computer Science fro...,"[bachelor, college, computer science]"
4,Neha Mugghala.docx,Bachelors in Computer Science and Information ...,"[computer science, university]"


### Experience extraction

In [70]:
EXPERIENCE_KEYWORDS = [
    "intern", "internship", "developer", "engineer", "analyst",
    "manager", "assistant", "research", "backend", "frontend",
    "full stack", "data analyst", "software engineer", "project manager",
    "leadership", "team", "organization", "volunteer"
]

In [71]:
df_resumes["experience_keywords"] = df_resumes["experience"].apply(
    lambda x: extract_keywords(x, EXPERIENCE_KEYWORDS)
)

In [72]:
df_resumes[["filename", "experience_keywords"]].head(10)

,filename,experience_keywords
0,Venkat_BA.docx,[]
1,Shashank.docx,"[analyst, assistant, leadership, research, team]"
2,Anudeep N_Sr Java Developer.docx,"[backend, developer, frontend, manager, team]"
3,ram nandyala.docx,"[developer, frontend, full stack, organization..."
4,Neha Mugghala.docx,"[developer, leadership, team]"
5,chenna kesava.docx,"[backend, developer, manager, organization, team]"
6,Utthan Silawal12.docx,"[analyst, data analyst, developer, leadership,..."
7,Yohan BSA.docx,"[analyst, data analyst, developer, manager, or..."
8,Madhu_BA_AW.DOCX,"[analyst, team]"
9,KIRAN KUMAR.docx,"[backend, developer, full stack, team]"


### Summary of extraction

In [77]:
df_resumes["num_skills"] = df_resumes["extracted_skills"].apply(len)
df_resumes["num_education_keywords"] = df_resumes["education_keywords"].apply(len)
df_resumes["num_experience_keywords"] = df_resumes["experience_keywords"].apply(len)
df_resumes["num_project_keywords"] = df_resumes["project_keywords"].apply(len)

In [78]:
df_resumes[[
    "filename",
    "email",
    "phone",
    "num_skills",
    "num_education_keywords",
    "num_experience_keywords"
]].head(10)

,filename,email,phone,num_skills,num_education_keywords,num_experience_keywords,num_project_keywords
0,Venkat_BA.docx,vhealthba@gmail.com,314-662-2902,2,2,0,0
1,Shashank.docx,Shashank.tiwari44@gmail.com,650) 600-1785,2,6,5,0
2,Anudeep N_Sr Java Developer.docx,anudeepreddynallamada@gmail.com,,13,2,5,0
3,ram nandyala.docx,NANDYALARAMAKARTHIK1@GMAIL.COM,414)-858-6880,14,3,5,0
4,Neha Mugghala.docx,sny.java@gmail.com,,17,2,3,0
5,chenna kesava.docx,chennakesava231@gmail.com,469-677-7837,17,0,5,0
6,Utthan Silawal12.docx,usilawal123@gmail.com,707) 653-6269,5,0,7,0
7,Yohan BSA.docx,,,8,0,6,0
8,Madhu_BA_AW.DOCX,bnsmadhavi@gmail.com,908-313-0336,10,0,2,0
9,KIRAN KUMAR.docx,kumarjava174@gmail.com,510-770-6277,15,1,4,0


## Job Description Parsing

In [79]:
job_description = """
About the role

We are looking for an experienced Software Engineer to join our team at Formulatrix Indonesia, a leading technology company based in Salatiga, Central Java. In this full-time role, you will play a crucial part in the development and implementation of cutting-edge software solutions that power our innovative products.

What you'll be doing

Design, develop and maintain high-quality, scalable software applications and systems.

Collaborate with cross-functional teams to understand business requirements and translate them into technical solutions.

Write clean, efficient and well-documented code using the latest software engineering best practices.

Identify and resolve complex technical issues, bugs and bottlenecks.

Contribute to the continuous improvement of our software development processes and tool.

What we're looking for

Bachelor’s degree in Computer Science, Information Technology, Electronics Engineering, or equivalent practical experience.

At least 1 year of experience as a Software Engineer or Developer, preferably in a similar role.

Strong proficiency in at least one object-oriented programming language (C#, C++, Python, Java, or similar).

Solid understanding of object-oriented programming, software architecture, and design patterns.

Experience with the full software development lifecycle.

Familiarity with unit testing, debugging, and version control systems.

Good understanding of software performance, scalability, and maintainability.

Willing to work in Salatiga / Bandung / Semarang (depending on placement).

Soft skills:

Strong analytical and problem-solving skills.

Curious, proactive, and eager to learn new technologies.

Highly motivated and a strong team player.

Clear and effective communication skills in English.

Detail-oriented with the ability to adapt and prioritize in a fast-paced R&D environment.

Nice to have:

Hands-on experience with robotics hardware, troubleshooting, and system debugging.

Hands-on experience developing robotics systems, including kinematics, control, and path planning.

Experience building and deploying neural networks for computer vision.

Strong knowledge of computer vision algorithms, optics, imaging systems, and familiarity with ML/CV frameworks (TensorFlow, PyTorch, OpenCV).

Proficient in Linux on SBCs (NVIDIA Jetson, Raspberry Pi) with a strong interest in research and emerging technologies.

Preferrable (one or more): Electronics Engineering background

What we offer

Enjoy working in a slow-living city with clean air, fresh water from mountain spring, beautiful natural sceneries, away from pollution and traffic-jam in large cities.

BPJS Health & Employment Insurance

Additional Health Insurance

Performance Bonus

Religious Holiday Allowance (THR)

Flexible Work Arrangements

Paid Time Off (eligible after 3 months of employment)

Casual Dress Code (e.g., T-shirts)

Office facilities that support professional productivity as well as social interaction and collaboration
"""

In [83]:
jd_clean_text = clean_resume_text(job_description)
print(jd_clean_text[:1000], "...")

about the role we are looking for an experienced software engineer to join our team at formulatrix indonesia a leading technology company based in salatiga central java. in this full-time role you will play a crucial part in the development and implementation of cutting-edge software solutions that power our innovative products. what you ll be doing design develop and maintain high-quality scalable software applications and systems. collaborate with cross-functional teams to understand business requirements and translate them into technical solutions. write clean efficient and well-documented code using the latest software engineering best practices. identify and resolve complex technical issues bugs and bottlenecks. contribute to the continuous improvement of our software development processes and tool. what we re looking for bachelor s degree in computer science information technology electronics engineering or equivalent practical experience. at least 1 year of experience as a softw

In [86]:
jd_extracted_skills = extract_skills(jd_clean_text, SKILL_KEYWORDS)
jd_education_keywords = extract_keywords(jd_clean_text, EDUCATION_KEYWORDS)
jd_experience_keywords = extract_keywords(jd_clean_text, EXPERIENCE_KEYWORDS)

jd_features = {
    "raw_text": job_description,
    "clean_text": jd_clean_text,
    "skills": jd_extracted_skills,
    "education_keywords": jd_education_keywords,
    "experience_keywords": jd_experience_keywords
}

jd_features

{'raw_text': "\nAbout the role\n\nWe are looking for an experienced Software Engineer to join our team at Formulatrix Indonesia, a leading technology company based in Salatiga, Central Java. In this full-time role, you will play a crucial part in the development and implementation of cutting-edge software solutions that power our innovative products.\n\nWhat you'll be doing\n\nDesign, develop and maintain high-quality, scalable software applications and systems.\n\nCollaborate with cross-functional teams to understand business requirements and translate them into technical solutions.\n\nWrite clean, efficient and well-documented code using the latest software engineering best practices.\n\nIdentify and resolve complex technical issues, bugs and bottlenecks.\n\nContribute to the continuous improvement of our software development processes and tool.\n\nWhat we're looking for\n\nBachelor’s degree in Computer Science, Information Technology, Electronics Engineering, or equivalent practical

In [88]:
print("="*80)
print("JOB DESCRIPTION SKILLS")
print("="*80)
print(jd_features["skills"])

print("\n" + "="*80)
print("JOB DESCRIPTION EDUCATION KEYWORDS")
print("="*80)
print(jd_features["education_keywords"])

print("\n" + "="*80)
print("JOB DESCRIPTION EXPERIENCE KEYWORDS")
print("="*80)
print(jd_features["experience_keywords"])

JOB DESCRIPTION SKILLS
['c', 'computer vision', 'java', 'linux', 'opencv', 'python', 'pytorch', 'r', 'spring', 'tensorflow']

JOB DESCRIPTION EDUCATION KEYWORDS
['bachelor', 'computer science', 'software engineering']

JOB DESCRIPTION EXPERIENCE KEYWORDS
['developer', 'engineer', 'research', 'software engineer', 'team']


## Matching & Ranking

### Match score

In [89]:
def compute_overlap_score(resume_items, jd_items):
    """
    Compute overlap ratio between resume feature list and JD feature list.
    Returns score between 0 and 1.
    """
    resume_set = set([str(x).lower() for x in resume_items])
    jd_set = set([str(x).lower() for x in jd_items])

    if len(jd_set) == 0:
        return 0.0

    matched = resume_set.intersection(jd_set)
    return len(matched) / len(jd_set)

In [90]:
# Compute skill match score
df_resumes["skill_match_score"] = df_resumes["extracted_skills"].apply(
    lambda x: compute_overlap_score(x, jd_features["skills"])
)

# Compute education match score
df_resumes["education_match_score"] = df_resumes["education_keywords"].apply(
    lambda x: compute_overlap_score(x, jd_features["education_keywords"])
)

# Compute experience match score
df_resumes["experience_match_score"] = df_resumes["experience_keywords"].apply(
    lambda x: compute_overlap_score(x, jd_features["experience_keywords"])
)